# Rubin / LSSTCam Stacked PSF Stars & Shapelet Library

This notebook is a self-contained guide to two datasets released alongside the Rubin DP2 image-quality study:

| Path | Contents |
|------|----------|
| `data/stamps_stack/` | Per-visit stacked PSF star stamps (32×32 px, 0.2″/px) |
| `data/shapelet_bvec/` | Pre-fitted bmax=6 Shapelet coefficient libraries (per band) |

**Sections**
1. Exploring the stamp files
2. `ShapeletPSFLibrary` — loading and drawing PSFs
3. FWHM and non-Gaussian shapelet power
4. Visualising star vs shapelet reconstruction vs residual
5. Lookup by `visit_id` + `raft_id`

## Setup

In [ ]:
import numpy as np
import galsim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import glob
import math

# ── paths ──────────────────────────────────────────────────────────────────
DATA_ROOT   = Path('/sdf/data/rubin/user/ztq1996/psf-rubin/psf_zernike/data')
STAMPS_DIR  = DATA_ROOT / 'stamps_stack'
BVEC_DIR    = DATA_ROOT / 'shapelet_bvec'

PIXEL_SCALE = 0.2    # arcsec / pixel
BMAX        = 6      # shapelet order used for all libraries
N_COEFFS    = (BMAX + 1) * (BMAX + 2) // 2   # 28

stamp_files = sorted(glob.glob(str(STAMPS_DIR / 'stamps_*.npz')))
bvec_files  = sorted(glob.glob(str(BVEC_DIR  / 'shapelet_*.npz')))
print(f'Stamp files : {len(stamp_files)}')
print(f'Bvec libs   : {len(bvec_files)}')
print(f'Example stamp file : {Path(stamp_files[0]).name}')
print(f'Example bvec  file : {Path(bvec_files[0]).name}')

## 1  Stamp files — `data/stamps_stack/`

Each file covers **one visit** and is named `stamps_{visit_id}.npz`.

| Array | Shape | Description |
|-------|-------|-------------|
| `stamps` | (N_rafts, 32, 32) float32 | Stacked PSF star image per raft |
| `visit` | (N_rafts,) int64 | Visit ID (repeated for all rafts) |
| `raft` | (N_rafts,) str | Raft name, e.g. `'R22'` |
| `detector` | (N_rafts,) int32 | Detector number for that raft |
| `band` | scalar str | Photometric band (`u/g/r/i/z/y`) |
| `cx`, `cy` | (N_rafts,) float | Focal-plane centre of the stacked stamp |
| `n_stack` | (N_rafts,) int | Number of stars stacked per raft |

In [ ]:
# Load one stamp file and inspect
d = np.load(stamp_files[0])
print('visit_id :', int(d['visit'][0]))
print('band     :', str(d['band']))
print('rafts    :', list(d['raft']))
print('stamps   :', d['stamps'].shape, d['stamps'].dtype)
print('n_stack  :', d['n_stack'])

# Show all rafts in this visit
n_rafts = d['stamps'].shape[0]
fig, axes = plt.subplots(2, math.ceil(n_rafts / 2), figsize=(16, 5))
for ax, i in zip(axes.flat, range(n_rafts)):
    vmax = np.nanpercentile(d['stamps'][i], 99.5)
    ax.imshow(d['stamps'][i], origin='lower', cmap='viridis', vmax=vmax)
    ax.set_title(f'{d["raft"][i]}  det={d["detector"][i]}\nn_stack={d["n_stack"][i]}', fontsize=7)
    ax.axis('off')
for ax in axes.flat[n_rafts:]:
    ax.axis('off')
fig.suptitle(f'All raft stamps — visit {int(d["visit"][0])}  band={d["band"]}', fontsize=11)
plt.tight_layout()
plt.show()

## 2  Shapelet bvec libraries — `data/shapelet_bvec/`

Libraries are named `shapelet_{band}_all.npz` (all DP2 visits) and `shapelet_{band}_post20251101.npz` (visits from 2025-11-01 onwards).

Each entry is a bmax=6 Shapelet fit (28 coefficients) to one raft's stacked PSF star.

| Array | Shape | Description |
|-------|-------|-------------|
| `bvec` | (N, 28) float64 | Shapelet coefficient vectors |
| `sigma` | (N,) float64 | HSM adaptive-moment scale σ [arcsec]; FWHM ≈ 2.355σ |
| `visit` | (N,) int64 | Visit ID |
| `detector` | (N,) int32 | Detector ID (use with the stamp file to recover `raft`) |
| `bmax` | scalar | Shapelet order (6) |
| `band` | scalar str | Photometric band |

The `ShapeletPSFLibrary` class below wraps these arrays and provides helpers to draw GalSim PSF images.

In [ ]:
class ShapeletPSFLibrary:
    """Load a pre-fitted shapelet bvec library and reconstruct GalSim PSF images.

    Parameters
    ----------
    npz_path : str or Path
        Path to a ``shapelet_{band}_*.npz`` file.

    Quick start
    -----------
    lib = ShapeletPSFLibrary('data/shapelet_bvec/shapelet_r_all.npz')
    psf = lib.get_psf(42)           # galsim.Shapelet at index 42
    img = lib.draw_psf(psf, n=32)   # draw onto a 32×32 image
    """

    def __init__(self, npz_path):
        data           = np.load(npz_path)
        self.bvec_all  = data['bvec']       # (N, 28) float64
        self.sigma_all = data['sigma']      # (N,)    arcsec
        self.bmax      = int(data['bmax'])
        self.band      = str(data['band'])
        self.visit     = data['visit']      # (N,) int64
        self.detector  = data['detector']   # (N,) int32
        self.npz_path  = Path(npz_path)
        self._n        = len(self.sigma_all)
        print(f'Loaded {self._n:,} PSFs | band={self.band} '
              f'bmax={self.bmax} n_coeffs={self.bvec_all.shape[1]} '
              f'from {self.npz_path.name}')

    def get_psf(self, idx):
        """Return the shapelet PSF at *idx* as a galsim.Shapelet."""
        return galsim.Shapelet(float(self.sigma_all[idx]), self.bmax,
                               self.bvec_all[idx])

    def sample_psf(self, rng=None):
        """Return a randomly chosen galsim.Shapelet from the library."""
        if rng is None:
            rng = np.random.default_rng()
        return self.get_psf(int(rng.integers(0, self._n)))

    def draw_psf(self, psf, n=32, pixel_scale=PIXEL_SCALE):
        """Draw *psf* onto an n×n image and return the pixel array."""
        img = galsim.Image(n, n, scale=pixel_scale)
        psf.drawImage(image=img, method='sb')
        return img.array

    def __len__(self):  return self._n
    def __repr__(self):
        return (f'ShapeletPSFLibrary(band={self.band!r}, n={self._n:,}, '
                f'bmax={self.bmax}, source={self.npz_path.name!r})')

### Load a library and draw random PSFs

In [ ]:
lib = ShapeletPSFLibrary(BVEC_DIR / 'shapelet_r_all.npz')

rng = np.random.default_rng(7)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax in axes.flat:
    idx  = int(rng.integers(0, len(lib)))
    psf  = lib.get_psf(idx)
    img  = lib.draw_psf(psf, n=32)
    fwhm = 2.355 * lib.sigma_all[idx]
    ax.imshow(img, origin='lower', cmap='viridis',
              vmax=np.nanpercentile(img, 99.5))
    ax.set_title(f'visit {lib.visit[idx]}\nFWHM={fwhm:.2f}"', fontsize=7)
    ax.axis('off')
fig.suptitle(f'10 random PSFs from {lib}', fontsize=10)
plt.tight_layout()
plt.show()

## 3  FWHM and non-Gaussian shapelet power

Two derived quantities computed directly from the library arrays:

```python
fwhm_arcsec       = 2.355 * lib.sigma_all          # FWHM in arcsec
nongaussian_power = np.sum(lib.bvec_all[:, 6:]**2, axis=1)
```

`sigma` is the adaptive-moment scale in arcsec returned by `FindAdaptiveMom`. FWHM = 2.355 σ is exact for a Gaussian.

`bvec[:, 6:]` drops the first 6 coefficients (indices 0–5 = Gaussian b₀₀, centroid/dipole n=1, ellipticity/quadrupole n=2, and radial-size n=2) and sums the squared remaining terms — everything from n≥3 (trefoil, octopole, hexapole, …). Large values indicate PSFs with significant higher-order non-Gaussian structure.

In [ ]:
fwhm_arcsec       = 2.355 * lib.sigma_all
nongaussian_power = np.sum(lib.bvec_all[:, 6:]**2, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# FWHM histogram
axes[0].hist(fwhm_arcsec, bins=60, color='steelblue', edgecolor='k', lw=0.3)
axes[0].set_xlabel('FWHM (arcsec)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('FWHM distribution', fontsize=11)
axes[0].axvline(np.median(fwhm_arcsec), color='red', lw=1.5,
                label=f'median = {np.median(fwhm_arcsec):.3f}"')
axes[0].legend(fontsize=9)

# Non-Gaussian power histogram
axes[1].hist(nongaussian_power, bins=60, range = (0,0.00003),
             color='tomato', edgecolor='k', lw=0.3)
axes[1].set_xlabel('Non-Gaussian power  Σ bvec[6:]²', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('Non-Gaussian power distribution', fontsize=11)

# FWHM vs non-Gaussian power scatter
sc = axes[2].scatter(fwhm_arcsec, nongaussian_power,
                     s=2, alpha=0.3, c='steelblue', rasterized=True)
axes[2].set_xlabel('FWHM (arcsec)', fontsize=11)
axes[2].set_ylabel('Non-Gaussian power', fontsize=11)
axes[2].set_title('FWHM vs non-Gaussian power', fontsize=11)

plt.suptitle(f'Band {lib.band}  |  {len(lib):,} PSFs', fontsize=12)
plt.tight_layout()
plt.show()

## 4  Star vs shapelet reconstruction vs residual

The `collect_rows` helper matches library indices back to their raw stamp files (via `visit` + `detector`), draws the shapelet model, and returns the triplet. `plot_rows` arranges them in a compact two-column grid.

In [ ]:
def collect_rows(indices, lib, stamps_dir, n=10):
    """For each library index load the raw stamp, reconstruct, and return rows."""
    rows = []
    for idx in indices:
        if len(rows) >= n:
            break
        visit    = int(lib.visit[idx])
        detector = int(lib.detector[idx])
        stamp_file = Path(stamps_dir) / f'stamps_{visit}.npz'
        if not stamp_file.exists():
            continue
        d    = np.load(stamp_file)
        mask = d['detector'] == detector
        if not mask.any():
            continue
        i        = int(np.where(mask)[0][0])
        star_img = d['stamps'][i].astype(np.float64)
        raft     = str(d['raft'][i])
        model    = lib.draw_psf(lib.get_psf(idx), n=star_img.shape[0])
        rows.append(dict(
            visit=visit, raft=raft,
            fwhm=2.355 * lib.sigma_all[idx],
            power=np.sum(lib.bvec_all[idx, 6:]**2),
            star=star_img, model=model, residual=star_img - model,
        ))
    return rows


def plot_rows(rows, title=''):
    """Two-column grid: left half = first ceil(n/2) rows, right half = remainder."""
    n    = len(rows)
    half = math.ceil(n / 2)
    fig, axes = plt.subplots(half, 6, figsize=(18, half * 2.8))
    if half == 1:
        axes = axes[np.newaxis, :]
    col_titles = ['Stacked star', 'Shapelet recon', 'Residual']
    for col_off in (0, 3):
        for j, t in enumerate(col_titles):
            axes[0, col_off + j].set_title(t, fontsize=10, fontweight='bold')
    for row_idx, r in enumerate(rows):
        group   = row_idx // half
        sub_row = row_idx % half
        col_off = group * 3
        axs = [axes[sub_row, col_off + j] for j in range(3)]
        label = (f'visit {r["visit"]}\nraft  {r["raft"]}'
                 f'\nFWHM={r["fwhm"]:.2f}"\npow={r["power"]:.2e}')
        vmax = np.nanpercentile(r['star'], 99.5)
        axs[0].imshow(r['star'], origin='lower', cmap='viridis', vmax=vmax)
        axs[0].text(0.03, 0.97, label, transform=axs[0].transAxes,
                    fontsize=6, va='top', ha='left', color='white',
                    bbox=dict(facecolor='black', alpha=0.55, pad=2, boxstyle='round'))
        vmax = np.nanpercentile(r['model'], 99.5)
        axs[1].imshow(r['model'], origin='lower', cmap='viridis', vmax=vmax)
        vlim = max(np.nanpercentile(np.abs(r['residual']), 99), 1e-10)
        axs[2].imshow(r['residual'], origin='lower', cmap='RdBu_r',
                      vmin=-vlim, vmax=vlim)
        for ax in axs:
            ax.axis('off')
    if n % 2 == 1:
        for j in range(3):
            axes[-1, 3 + j].axis('off')
    if title:
        fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

### 10 random PSFs from the library

In [ ]:
rng  = np.random.default_rng(42)
idxs = rng.choice(len(lib), size=200, replace=False)   # pool to cover missing files
rows = collect_rows(idxs, lib, STAMPS_DIR, n=10)
print(f'Collected {len(rows)} rows')
plot_rows(rows, title=f'Random sample — band {lib.band}')

### Filter by FWHM or non-Gaussian power

Pass any boolean mask over the library index to select subsets.

In [ ]:
# e.g. best-seeing (FWHM < 0.7") vs. worst-seeing (FWHM > 1.1")
good_seeing = np.where(fwhm_arcsec < 0.7)[0]
bad_seeing  = np.where(fwhm_arcsec > 1.1)[0]
print(f'FWHM < 0.7" : {len(good_seeing):,} entries')
print(f'FWHM > 1.1" : {len(bad_seeing):,} entries')

rng = np.random.default_rng(1)
for pool, label in [(good_seeing, 'good seeing  FWHM < 0.7"'),
                    (bad_seeing,  'bad seeing   FWHM > 1.1"')]:
    sample = rng.choice(pool, size=min(100, len(pool)), replace=False)
    rows   = collect_rows(sample, lib, STAMPS_DIR, n=6)
    if rows:
        plot_rows(rows, title=label)

## 5  Lookup by `visit_id` + `raft_id`

The library stores `visit` and `detector`.  To retrieve a specific raft, load the stamp file for that visit (to map `raft → detector`), then match `visit + detector` in the library.

In [ ]:
def get_by_visit_raft(lib, stamps_dir, visit_id, raft_id):
    """Return the star / shapelet / residual triplet for one visit+raft.

    Parameters
    ----------
    lib        : ShapeletPSFLibrary
    stamps_dir : path to stamps_stack/
    visit_id   : int, e.g. 2025042400172
    raft_id    : str, e.g. 'R22'

    Returns
    -------
    dict with keys: visit, raft, detector, n_stack, fwhm, power,
                    star, model, residual, bvec, sigma
    """
    stamp_file = Path(stamps_dir) / f'stamps_{visit_id}.npz'
    if not stamp_file.exists():
        raise FileNotFoundError(f'No stamp file for visit {visit_id}')
    d    = np.load(stamp_file)
    rmask = np.array([str(r) for r in d['raft']]) == str(raft_id)
    if not rmask.any():
        available = list(d['raft'])
        raise ValueError(f'Raft {raft_id!r} not in visit {visit_id}. '
                         f'Available: {available}')
    si       = int(np.where(rmask)[0][0])
    detector = int(d['detector'][si])
    star_img = d['stamps'][si].astype(np.float64)

    lmask = (lib.visit == visit_id) & (lib.detector == detector)
    if not lmask.any():
        raise ValueError(f'No library entry for visit={visit_id} raft={raft_id} '
                         f'(detector={detector})')
    li    = int(np.where(lmask)[0][0])
    model = lib.draw_psf(lib.get_psf(li), n=star_img.shape[0])

    return dict(
        visit=visit_id, raft=raft_id, detector=detector,
        n_stack=int(d['n_stack'][si]),
        fwhm=2.355 * lib.sigma_all[li],
        power=float(np.sum(lib.bvec_all[li, 6:]**2)),
        bvec=lib.bvec_all[li].copy(),
        sigma=float(lib.sigma_all[li]),
        star=star_img, model=model, residual=star_img - model,
    )


def plot_single(entry):
    """Show star / shapelet / residual for one entry with metadata."""
    fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
    for ax, t in zip(axes, ['Stacked star', 'Shapelet recon', 'Residual']):
        ax.set_title(t, fontsize=11, fontweight='bold')
    vmax = np.nanpercentile(entry['star'], 99.5)
    axes[0].imshow(entry['star'],  origin='lower', cmap='viridis', vmax=vmax)
    vmax = np.nanpercentile(entry['model'], 99.5)
    axes[1].imshow(entry['model'], origin='lower', cmap='viridis', vmax=vmax)
    vlim = max(np.nanpercentile(np.abs(entry['residual']), 99), 1e-10)
    axes[2].imshow(entry['residual'], origin='lower', cmap='RdBu_r',
                   vmin=-vlim, vmax=vlim)
    for ax in axes:
        ax.axis('off')
    fig.suptitle(
        f'visit={entry["visit"]}  raft={entry["raft"]}  '
        f'det={entry["detector"]}  n_stack={entry["n_stack"]}  '
        f'FWHM={entry["fwhm"]:.3f}"  '
        f'non-Gaussian power={entry["power"]:.2e}',
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()

### Example: query a specific visit and raft

In [ ]:
# Pick any visit that appears in the library
example_visit = int(lib.visit[0])
example_raft  = 'R22'   # change to any valid raft for that visit

entry = get_by_visit_raft(lib, STAMPS_DIR, example_visit, example_raft)
print(f'visit    : {entry["visit"]}')
print(f'raft     : {entry["raft"]}  detector={entry["detector"]}')
print(f'n_stack  : {entry["n_stack"]} stars stacked')
print(f'FWHM     : {entry["fwhm"]:.3f} arcsec')
print(f'non-Gauss power : {entry["power"]:.3e}')
print(f'bvec[0] (Gaussian amplitude) : {entry["bvec"][0]:.4f}')

plot_single(entry)

### List all visits available for a given raft

Because the library stores `visit` and `detector`, you can filter programmatically.

In [ ]:
# What visits in the library have a R22 stamp?
target_raft = 'R22'

# Find detector number(s) corresponding to this raft across all stamp files
raft_detector_map = {}
for f in stamp_files[:50]:    # scan first 50 files to build the map
    d = np.load(f)
    rafts = np.array([str(r) for r in d['raft']])
    hits  = np.where(rafts == target_raft)[0]
    if hits.size:
        det = int(d['detector'][hits[0]])
        raft_detector_map[det] = target_raft  # detector is the same across visits

if raft_detector_map:
    det = list(raft_detector_map)[0]
    visit_mask = lib.detector == det
    visits_for_raft = lib.visit[visit_mask]
    print(f'Raft {target_raft} → detector {det}')
    print(f'Found {len(visits_for_raft)} library entries for this raft')
    print(f'First 10 visit IDs: {sorted(set(visits_for_raft))[:10]}')
else:
    print(f'Raft {target_raft} not found in first 50 stamp files')

## Appendix — Shapelet coefficient index reference (bmax=6)

The 28 bvec entries are ordered by total order n = p+q, then by m = p−q descending. Each (p,q) with p ≠ q occupies **two** consecutive slots (Re, Im); p = q occupies **one**.

```
idx   (p,q)  n  m   physical meaning
  0   (0,0)  0  0   Gaussian
  1   (1,0)  1  1   centroid / dipole  Re
  2   (1,0)  1  1   centroid / dipole  Im
  3   (2,0)  2  2   ellipticity        Re
  4   (2,0)  2  2   ellipticity        Im
  5   (1,1)  2  0   radial size correction
  6   (3,0)  3  3   trefoil            Re   ┐
  7   (3,0)  3  3   trefoil            Im   │
  8   (2,1)  3  1   dipole n=3         Re   │ non-Gaussian
  9   (2,1)  3  1   dipole n=3         Im   │ bvec[6:]
 10   (4,0)  4  4   octopole           Re   │
 ...                                        │
 27   (3,3)  6  0   radial n=6         Re   ┘
```

`nongaussian_power = np.sum(lib.bvec_all[:, 6:]**2, axis=1)` sums everything from index 6 onwards (n≥3 terms: trefoil, octopole, hexapole, and their radial companions).